In [10]:
!pip install pandas 
!pip install openpyxl
import pandas as pd
import json

df = pd.read_excel("/Users/olivia/Desktop/Thomson-reconcile_v4.xlsx")

In [ ]:
import json
import pandas as pd

geojsonData = {
    "type": "FeatureCollection",
    "features": []
}

unique_cities = {}
unique_edges = {}
cols = df.columns

for idx, row in df.iterrows():
    # Helper for safe data access
    get_val = lambda col: str(row[col]).strip() if col in cols and row[col] is not None and str(row[col]) != 'nan' else ""

    from_name = get_val("from_reconcile")
    to_name = get_val("to_reconcile")
    if not from_name or not to_name: continue 

    from_id = f"city-{from_name}".replace(" ", "-")
    to_id = f"city-{to_name}".replace(" ", "-")

    # --- 1. PROCESS CITIES (NODES) ---
    city_configs = [
        (from_id, from_name, "from_coordinates1_lon", "from_coordinates1_lat", "from_image_1_URL"),
        (to_id, to_name, "to_coordinates1_lon", "to_coordinates1_lat", "to_image_1_URL")
    ]

    for cid, cname, lon_col, lat_col, img_col in city_configs:
        if cid not in unique_cities: # avoid repetition of cities
            lon = row[lon_col] if lon_col in cols else 0
            lat = row[lat_col] if lat_col in cols else 0
            unique_cities[cid] = {
                "type": "Feature",
                "id": cid,
                "geometry": {"type": "Point", "coordinates": [lon, lat]},
                "properties": {"name": cname, "type": "city", "media": []}
            }
        
        # Add modern city image (NO CAPTION HERE)
        c_img = get_val(img_col)
        if c_img and not any(m['image'] == c_img for m in unique_cities[cid]["properties"]["media"]):
            unique_cities[cid]["properties"]["media"].append({
                "image": c_img, 
                "text": "" 
                # Caption omitted for modern city images
            })

    # --- 2. PROCESS EDGES (LINESTRINGS) ---
    current_img = get_val("image_url")
    current_text = get_val("context_full")
    current_cap = get_val("Column1") # This column is a combination of caption, year, and book page

    if from_id == to_id:
        # Self-loop: Add historical data (WITH caption) to the city point
        if current_img:
            unique_cities[from_id]["properties"]["media"].append({
                "image": current_img,
                "text": current_text,
                "caption": current_cap
            })
    else:
        # Standard Edge: Add historical data (WITH caption)
        edge_id = f"edge-{from_name}-{to_name}".replace(" ", "-")
        if edge_id not in unique_edges: # avoid repetition of edges
            # Safely grab coordinates for the line
            coords = [
                [row.get("from_coordinates1_lon", 0), row.get("from_coordinates1_lat", 0)],
                [row.get("to_coordinates1_lon", 0), row.get("to_coordinates1_lat", 0)]
            ]
            unique_edges[edge_id] = {
                "type": "Feature",
                "id": edge_id,
                "geometry": {"type": "LineString", "coordinates": coords},
                "properties": {
                    "type": "boundary", "from": from_name, "to": to_name, "media": []
                }
            }
        
        if current_img:
            unique_edges[edge_id]["properties"]["media"].append({
                "image": current_img,
                "text": current_text,
                "caption": current_cap
            })

geojsonData["features"] = list(unique_cities.values()) + list(unique_edges.values())

with open("/Users/olivia/Desktop/geojson_output.json", "w") as f:    
    json.dump(geojsonData, f, indent=4)